## tl;dr

2026-08-10~12 광주 생산량은 `DB_생산실적.xlsx`에 36,247 / 23,250 / 32,789 kg가 있으나, `RawDB_생산실적.xlsx` 최신 우선 병합이 각 날짜를 0 kg로 덮어 `DB_에너지.xlsx`도 0 kg가 된다. 검증 가능한 8월 1~14일 합계 차이는 92,286 kg이다.

## Context & Methods

RPA와 동일한 `energy_builder` 로더를 사용해 공장 코드 F30의 일자별 합계를 재현하고, 광주 에너지 시트의 `믹스생산량[kg]`과 비교한다. 모든 파일은 읽기 전용으로 연다.

### Key Assumptions

- 비교 단위는 공장 F30 × 일자 × kg이다.
- `DB_생산실적.xlsx`의 `daily` 시트를 기준값 후보로 사용한다.
- 2026-08-15~17은 `DB_생산실적` 값이 없으므로 합계 검증에서 제외한다.

## Data

In [ ]:
from analysis.debug_gwangju_energy import reconcile
rows = reconcile()
rows

## Results

In [ ]:
mismatches = [row for row in rows if row['gap_kg'] not in (None, 0)]
verified = [row for row in rows if row['db_production_kg'] is not None]
db_total = sum(row['db_production_kg'] or 0 for row in verified)
energy_total = sum(row['db_energy_kg'] or 0 for row in verified)
print('mismatches:', mismatches)
print('verified_days:', len(verified))
print('db_total_kg:', db_total)
print('energy_total_kg:', energy_total)
print('gap_total_kg:', db_total - energy_total)

## Takeaways

- 직접 불일치는 2026-08-10~12 세 날짜에 집중된다.
- `merge_production_actuals()`가 `DB_생산실적` 위에 Raw 값을 무조건 덮어쓰며, Raw 로더가 기간 마커 내 누락 날짜를 0으로 채우는 조합이 원인이다.
- 같은 실행에서 `F30_상온` 조회가 `F30_냉장_FM`과 동일해 3회 거부되었고 기존 Raw 시트를 보존했다. 이 시트는 냉장과 같은 품목·값을 담고 있어 별도 상위 데이터 품질 문제다.